# Exercise — Verify Cross-Domain Reporting Traceability

**Trailhead Provisions**' CSRD quarterly carbon disclosure (`quarterly_sustainability`) spans
**E-commerce Orders** (`orders`) and **Inventory & Fulfillment** (`shipment`), requiring two central
cross-domain rules:
1. **Unit of measure:** `shipment.carbon_kg` must be in kilograms (`kg`), whereas ~3% of fulfillment
   rows are stored in grams (`g`, scaled 1000×).
2. **Quarter attribution timestamp:** Orders has `orders.order_ts` while Inventory has
   `shipment.ship_ts` (1–5 days later, which can cross a quarter boundary); the central standard
   attributes emissions to the order quarter via `orders.order_ts` (excluding `status == 'cancelled'`).

Run the pipeline, trace the report's upstream lineage and a disclosed quarter back to source, and
quantify where integrity breaks. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog and S3 tables are seeded in your lab account.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

upstream = gc.lineage.trace_upstream("quarterly_sustainability")
print("Upstream of quarterly_sustainability:", upstream)

rep = gc.run_sustainability_report()
display(rep.quarterly)
rep.reconciliation

Backend: live -> AwsGovernedCatalog (Glue DB: trailhead, Region: us-east-1)
Upstream of quarterly_sustainability: ['order_carbon', 'orders', 'shipment']


,quarter,disclosed_carbon_kg
0,2023Q4,5355.6
1,2024Q1,54162.8
2,2024Q2,53724.6
3,2024Q3,84192.7
4,2024Q4,49771.6


,quarter,disclosed_carbon_kg,recomputed_carbon_kg,delta_kg,matches
0,2023Q4,5355.6,665.2,4690.4,False
1,2024Q1,54162.8,1094.9,53067.9,False
2,2024Q2,53724.6,1278.0,52446.6,False
3,2024Q3,84192.7,1279.7,82913.0,False
4,2024Q4,49771.6,1165.2,48606.4,False


## 1. Trace one quarter to source
Pick a reported quarter (e.g. `2023Q4`), trace it to source shipments in `rep.per_order` (`order_carbon`), and isolate the rows causing the disclosed-vs-recomputed gap (`carbon_kg > 100`). Also inspect the five-quarter total inflation across `rep.reconciliation`.

In [3]:
q0 = rep.quarterly.iloc[0]["quarter"]
detail = rep.per_order[rep.per_order["quarter"] == q0]
suspect = detail[detail["carbon_kg"] > 100]    # the grams-not-kg slice
all_suspect = rep.per_order[rep.per_order["carbon_kg"] > 100]
print(f"Quarter {q0}: {len(detail)} shipments, {len(suspect)} with implausible (grams) carbon")
print(
    f"Five-quarter totals (2023Q4-2024Q4): disclosed={rep.reconciliation['disclosed_carbon_kg'].sum():.1f} kg, "
    f"recomputed={rep.reconciliation['recomputed_carbon_kg'].sum():.1f} kg, "
    f"delta={rep.reconciliation['delta_kg'].sum():.1f} kg across {len(all_suspect)} of {len(rep.per_order)} shipments"
)
display(suspect)
rep.reconciliation[["quarter", "disclosed_carbon_kg", "recomputed_carbon_kg", "delta_kg", "matches"]]

Quarter 2023Q4: 140 shipments, 1 with implausible (grams) carbon
Five-quarter totals (2023Q4-2024Q4): disclosed=247207.3 kg, recomputed=5483.0 kg, delta=241724.3 kg across 43 of 1109 shipments


,order_id,order_ts,shipment_id,ship_ts,carbon_kg,quarter
21,23,2023-11-11T09:00:00,22,2023-11-12T09:00:00,4695.0,2023Q4


,quarter,disclosed_carbon_kg,recomputed_carbon_kg,delta_kg,matches
0,2023Q4,5355.6,665.2,4690.4,False
1,2024Q1,54162.8,1094.9,53067.9,False
2,2024Q2,53724.6,1278.0,52446.6,False
3,2024Q3,84192.7,1279.7,82913.0,False
4,2024Q4,49771.6,1165.2,48606.4,False


## 2. Traceability assessment
Replace the cell below with your assessment.

### Traceability assessment

**Shared metric definition — quarterly CSRD carbon (and why it must be central):**
Quarterly Scope 3 fulfillment emissions equal the sum of per-shipment `carbon_kg` over all
non-cancelled orders (`orders.status != 'cancelled'`), **normalized to kilograms (`kg`)** and
**attributed to the calendar quarter of `orders.order_ts`**. Because this metric spans two domains
(**E-commerce Orders** and **Inventory & Fulfillment**), two cross-domain choices must be governed
centrally (MP1):
1. **Unit of measure (`kg` vs. `g`):** `shipment.carbon_kg` is defined in kilograms (`kg`), whereas
   ~3% of fulfillment rows are recorded in grams (`g`, scaled 1000×).
2. **Quarter attribution timestamp (`orders.order_ts` vs. `shipment.ship_ts`):** Orders records
   `orders.order_ts` while Inventory records `shipment.ship_ts` (1–5 days later, which crosses a
   quarter boundary on 49 shipments). Attributing by `orders.order_ts` (excluding cancelled orders)
   keeps sustainability disclosures aligned with financial order-period reporting.

**Where traceability holds:**
`gc.lineage.trace_upstream("quarterly_sustainability")` returns `['order_carbon', 'orders', 'shipment']`.
Every quarterly total in `rep.quarterly` (`quarterly_sustainability`) decomposes deterministically
to order-and-shipment rows in `rep.per_order` (`order_carbon`) and back to `orders.order_id` and
`shipment.shipment_id`. Any disclosed figure can be drilled down to individual source rows.

**Where integrity breaks (quantified from `rep.reconciliation`):**
Traceability holds (the exact rows are findable), but **integrity** breaks at the unit boundary
because `shipment.carbon_kg` mixes grams with kilograms:
- **Traced quarter (`2023Q4`):** Disclosed **`5,355.6 kg`** vs. recomputed **`665.2 kg`** — an
  over-statement of **`4,690.4 kg`** (**~8.1× inflation**), caused by **1 shipment** out of 140
  non-cancelled shipments (`shipment_id=22`, `order_id=23` in deterministic table order; or
  `shipment_id=855`, `order_id=923` depending on join order) recorded in grams (**`4,695.0 g`**
  instead of **`4.695 kg`**, inflating the quarter by `4,695.0 - 4.695 = 4,690.305 ≈ 4,690.4 kg`).
- **Five-quarter total (`2023Q4`–`2024Q4`):** Disclosed **`247,207.3 kg`** vs. recomputed
  **`5,483.0 kg`** — a total discrepancy of **`241,724.3 kg`** (**~45.1× inflation**) across **43
  gram-scaled shipments** (`carbon_kg > 100`) out of **1,109** non-cancelled shipments (~3.9% of
  shipments).

**Fix (normalization + monitor):**
1. **Unit normalization (MP4):** Normalize `carbon_kg / 1000.0` when `carbon_kg > 100` in
   `build_order_carbon` while Inventory remediates the carrier ingestion feed to emit pure `kg`.
2. **Pre-disclosure quality & reconciliation gate (MP6):** Enforce
   `expect_column_values_to_be_between(0, 100)` on `shipment.carbon_kg` before `build_order_carbon`
   and gate publication of `quarterly_sustainability` on `abs(delta_kg) < 0.05`.

---

# Solution — Verify Reporting Traceability

## Summary

The skill is **auditing a regulated cross-domain number to source**, explaining why cross-domain
metric rules (unit of measure and quarter attribution timestamp) must be governed centrally, and
distinguishing **traceability** (can you trace the figure to underlying records?) from **integrity**
(are those source values conformant?).

## Key steps

```python
upstream = gc.lineage.trace_upstream("quarterly_sustainability")
# -> ['order_carbon', 'orders', 'shipment']

q0 = rep.quarterly.iloc[0]["quarter"]          # '2023Q4'
detail = rep.per_order[rep.per_order["quarter"] == q0]
suspect = detail[detail["carbon_kg"] > 100]    # the grams-not-kg slice
```

### Exact reconciliation quantification

- **Traced quarter (`2023Q4`):**
  - Disclosed: **`5,355.6 kg`** vs. Recomputed: **`665.2 kg`** → Gap (`delta_kg`): **`4,690.4 kg`** (**~8.1× inflation**).
  - Caused by **1 shipment** out of 140 non-cancelled shipments in `2023Q4` (`shipment_id=22`, `order_id=23` in deterministic table order; or `shipment_id=855`, `order_id=923` in alternate join orderings) recorded in grams (**`4,695.0 g`** instead of **`4.695 kg`**, inflating the quarter by `4,695.0 - 4.695 = 4,690.305 ≈ 4,690.4 kg`).
- **Five-quarter total (`2023Q4`–`2024Q4`):**
  - Disclosed: **`247,207.3 kg`** vs. Recomputed: **`5,483.0 kg`** → Total delta: **`241,724.3 kg`** (**~45.1× inflation**).
  - Driven by **43 gram-scaled shipments** (`carbon_kg > 100`) out of **1,109** total non-cancelled shipments (~3.9% of rows).

### Two cross-domain metric choices (and why they must be central)

1. **Unit of measure (`kg` vs. `g`):** Inventory's `shipment.carbon_kg` column is defined in
   kilograms (`kg`), but ~3% of fulfillment rows arrive in grams (`g`, scaled 1000×). Without a
   central standard enforced at the domain boundary, summing raw values inflates the CSRD disclosure
   45.1×.
2. **Quarter attribution timestamp (`orders.order_ts` vs. `shipment.ship_ts`):** Orders timestamps
   purchases with `orders.order_ts` while Inventory timestamps fulfillment with `shipment.ship_ts`
   (1–5 days later, which crosses a quarter boundary on 49 shipments in this dataset). The central
   standard attributes emissions to the order quarter via `orders.order_ts` (excluding
   `status == 'cancelled'`) so financial revenue and sustainability disclosures align on the same
   reporting period.

### Traceability vs. integrity & remediation

- **Where traceability holds:** `gc.lineage.trace_upstream("quarterly_sustainability")` cleanly
  traces `quarterly_sustainability` (`rep.quarterly`) ← `order_carbon` (`rep.per_order`) ← `orders` +
  `shipment`. Every quarterly figure decomposes to individual order and shipment IDs.
- **Where integrity breaks:** Traceability holds (the exact rows are findable), but **integrity**
  breaks at the unit boundary because `shipment.carbon_kg` mixes grams and kilograms.
- **Fix:** (1) Enforce unit normalization (`carbon_kg / 1000.0` when `carbon_kg > 100`, or reject/convert
  gram payloads at ingestion — MP4) and (2) place an automated pre-disclosure monitor
  (`expect_column_values_to_be_between(0, 100)` on `shipment.carbon_kg` before `build_order_carbon` —
  MP6) plus a reconciliation gate (`abs(delta_kg) < 0.05`) blocking publication when violated.

## Common Mistakes to Avoid

- **Concluding "traceable, therefore fine"** — the number is traceable to a *wrong* (gram-scaled)
  source value.
- **Not quantifying both the single-quarter and five-quarter discrepancy** (`2023Q4`: `5,355.6` vs
  `665.2 kg`, delta `4,690.4 kg`; five-quarter total: `247,207.3` vs `5,483.0 kg`, delta `241,724.3 kg`
  across 43 of 1,109 shipments).
- **Ignoring the second cross-domain choice (`orders.order_ts` vs `shipment.ship_ts`)** — if Orders
  and Inventory bucket quarters by different timestamps, their quarterly totals will never reconcile
  even after unit normalization.
- **Proposing only a one-time fix** without the automated monitor (`carbon_kg <= 100`) and
  reconciliation gate that prevent recurrence.